# Photovoltaic Thermography — Binary Panel Classifier
Colab-ready training notebook: preprocess JSON polygons, crop panels, train a binary classifier (healthy vs defective), evaluate, and export.

## 0. Environment & Paths
Sets Colab/local mode, project root, and data roots.

In [ ]:
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    ENV = 'colab'
except ModuleNotFoundError:
    ENV = 'local'

if ENV == 'colab':
    PROJECT_ROOT = Path('/content/hafar-pv-maintenance')
    RAW_ROOT = Path('/content/data/raw_photovoltaic')
    PROCESSED_ROOT = Path('/content/data/photovoltaic-system-thermography')
    DRIVE_DATA_DIR = Path('/content/drive/MyDrive/Photovoltaic_dataset')
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
    RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
    PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'photovoltaic-system-thermography'
    DRIVE_DATA_DIR = None

print('ENV:', ENV)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_ROOT:', RAW_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)


## 1. Mount Drive & Install Deps
Mount Google Drive (Colab), ensure Python ≥3.10, install the project and training deps.

In [ ]:
if ENV == 'colab':
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

%cd $PROJECT_ROOT
!python --version

install_cmd = (
    'pip install --quiet -e .'
    ' torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2'
    ' pytorch-lightning==2.2.5 torchmetrics==1.3.2'
    ' albumentations==1.4.10 segmentation-models-pytorch==0.3.3'
    ' flirimageextractor flyr'
)
!$install_cmd

src_path = PROJECT_ROOT / 'src'
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
print('Deps installed; src on sys.path')


## 2. Unzip Dataset from Drive
Extract `Photovoltaic_dataset.zip` (adjust name if needed) into `RAW_ROOT`. JSON basenames must match their images.

In [ ]:
import zipfile

DATASET_ZIP_NAME = 'Photovoltaic_dataset.zip'
dataset_zip = (DRIVE_DATA_DIR / DATASET_ZIP_NAME) if DRIVE_DATA_DIR else PROJECT_ROOT / DATASET_ZIP_NAME

RAW_ROOT.mkdir(parents=True, exist_ok=True)
if not any(RAW_ROOT.rglob('*.json')):
    if not dataset_zip.exists():
        raise FileNotFoundError(f'Missing dataset archive: {dataset_zip}')
    with zipfile.ZipFile(dataset_zip) as zf:
        zf.extractall(RAW_ROOT)
    print(f'Extracted {dataset_zip} -> {RAW_ROOT}')
else:
    print('Raw data already present; skipping unzip')

json_count = len(list(RAW_ROOT.rglob('*.json')))
images_count = len(list(RAW_ROOT.rglob('*.jpg')))
print(f'Found {json_count} annotation files and {images_count} JPG images')


## 3. Preprocess to Panels & Manifests
Use `PhotovoltaicThermographyPreprocessor` to crop panels and build manifests with `defective` labels.

In [ ]:
from hafar_pv.data.photovoltaic_thermography import PhotovoltaicThermographyPreprocessor

preprocessor = PhotovoltaicThermographyPreprocessor(
    raw_root=RAW_ROOT,
    output_root=PROCESSED_ROOT,
    resize_panels_to=(224, 224),
    val_fraction=0.15,
    test_fraction=0.15,
    random_state=42,
)
preprocessor.run()
print('Preprocessing complete')


## 4. Manifest & Label Balance
Load manifest, check splits and class balance, derive `pos_weight` for loss.

In [ ]:
import pandas as pd
import torch

panels_manifest = pd.read_csv(PROCESSED_ROOT / 'metadata' / 'panels_manifest.csv')
panels_manifest['defective'] = panels_manifest['defective'].astype(int)
print('Split counts:', panels_manifest['split'].value_counts().to_dict())
print('Label balance:', panels_manifest['defective'].value_counts().to_dict())

pos = (panels_manifest['defective'] == 1).sum()
neg = (panels_manifest['defective'] == 0).sum()
pos_weight = torch.tensor([neg / max(pos, 1)])
print('pos_weight:', pos_weight.item())


## 5. Datasets & Dataloaders
PanelDataset on NPZ crops with augmentations and optional class balancing.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from hafar_pv.data.datasets import PanelDataset

def _manifest_to_paths(df: pd.DataFrame):
    return [PROCESSED_ROOT / Path(p) for p in df['npz_path']]

train_df = panels_manifest[panels_manifest['split'] == 'train']
val_df = panels_manifest[panels_manifest['split'] == 'val']
test_df = panels_manifest[panels_manifest['split'] == 'test']

to_3ch = transforms.Lambda(lambda t: t.repeat(3, 1, 1))
train_tfms = transforms.Compose([
    to_3ch,
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25]),
])
eval_tfms = transforms.Compose([
    to_3ch,
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25]),
])

train_ds = PanelDataset(_manifest_to_paths(train_df), transform=train_tfms, target_key='label')
val_ds = PanelDataset(_manifest_to_paths(val_df), transform=eval_tfms, target_key='label')
test_ds = PanelDataset(_manifest_to_paths(test_df), transform=eval_tfms, target_key='label')

if pos > 0:
    weights = [pos_weight.item() if lbl == 1 else 1.0 for lbl in train_df['defective']]
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
else:
    sampler = None

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, shuffle=sampler is None, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Train/Val/Test sizes:', len(train_ds), len(val_ds), len(test_ds))


## 6. Lightning Module (Binary Classifier)
ResNet18 backbone, `BCEWithLogitsLoss` with `pos_weight`, metrics AUROC/F1.

In [ ]:
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
import torchmetrics
from torchvision import models

class PanelClassifier(pl.LightningModule):
    def __init__(self, pos_weight: torch.Tensor):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        backbone.fc = nn.Linear(backbone.fc.in_features, 1)
        self.model = backbone
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.train_auc = torchmetrics.AUROC(task='binary')
        self.val_auc = torchmetrics.AUROC(task='binary')
        self.val_f1 = torchmetrics.F1Score(task='binary', threshold=0.5)
        self.test_auc = torchmetrics.AUROC(task='binary')
        self.test_f1 = torchmetrics.F1Score(task='binary', threshold=0.5)

    def forward(self, x):
        return self.model(x).squeeze(1)

    def _shared_step(self, batch, stage):
        logits = self(batch['image'])
        targets = batch['target'].squeeze(1)
        loss = self.criterion(logits, targets)
        probs = torch.sigmoid(logits)
        if stage == 'train':
            self.train_auc.update(probs, targets.int())
        elif stage == 'val':
            self.val_auc.update(probs, targets.int())
            self.val_f1.update(probs, targets.int())
        else:
            self.test_auc.update(probs, targets.int())
            self.test_f1.update(probs, targets.int())
        self.log(f'{stage}_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, 'val')

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, 'test')

    def on_train_epoch_end(self):
        self.log('train_auc', self.train_auc.compute(), prog_bar=True)
        self.train_auc.reset()

    def on_validation_epoch_end(self):
        self.log('val_auc', self.val_auc.compute(), prog_bar=True)
        self.log('val_f1', self.val_f1.compute(), prog_bar=True)
        self.val_auc.reset(); self.val_f1.reset()

    def on_test_epoch_end(self):
        self.log('test_auc', self.test_auc.compute(), prog_bar=True)
        self.log('test_f1', self.test_f1.compute(), prog_bar=True)
        self.test_auc.reset(); self.test_f1.reset()

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}
        }

model = PanelClassifier(pos_weight=pos_weight)
print('Model params (M):', sum(p.numel() for p in model.parameters()) / 1e6)


## 7. Train
Train with mixed precision and checkpoint best model to Drive/local.

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

ckpt_dir = Path('/content/drive/MyDrive/pv_checkpoints') if ENV == 'colab' else PROJECT_ROOT / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_cb = ModelCheckpoint(
    dirpath=ckpt_dir,
    filename='pv-panel-bin-{epoch:02d}-{val_auc:.3f}',
    save_top_k=1,
    monitor='val_auc',
    mode='max',
)
early_cb = EarlyStopping(monitor='val_auc', mode='max', patience=5)

trainer = pl.Trainer(
    max_epochs=30,
    precision='16-mixed',
    accelerator='auto',
    devices='auto',
    callbacks=[ckpt_cb, early_cb],
    log_every_n_steps=10,
)

trainer.fit(model, train_loader, val_loader)
print('Best checkpoint:', ckpt_cb.best_model_path)


## 8. Test & Export
Evaluate on test set and save TorchScript for deployment.

In [ ]:
best_ckpt = ckpt_cb.best_model_path or None
if best_ckpt:
    model = PanelClassifier.load_from_checkpoint(best_ckpt, pos_weight=pos_weight)

trainer.test(model, dataloaders=test_loader)

export_path = ckpt_dir / 'panel_classifier.ts'
model.eval().to('cpu')
example = torch.randn(1, 3, 224, 224)
traced = torch.jit.trace(model, example)
traced.save(export_path)
print('Saved TorchScript to', export_path)
